# 3. Faster R-CNN with Torchvision
A two-stage detector: first proposes regions, then classifies objects and refines boxes. Pretrained weights are used for a quick demo.

In [ ]:
!pip -q install torch torchvision matplotlib pillow requests

In [ ]:
import torch, torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from PIL import Image
import requests, matplotlib.pyplot as plt
from io import BytesIO
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights).to(device).eval()
preprocess = weights.transforms()
url = "https://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
x = preprocess(image).to(device)
with torch.no_grad():
    pred = model([x])[0]
categories = weights.meta["categories"]
print("Device:", device, "| detections over 0.6 confidence:", int((pred["scores"] > .6).sum()))

In [ ]:
import matplotlib.patches as patches
fig, ax = plt.subplots(figsize=(12,8)); ax.imshow(image)
for box, label, score in zip(pred["boxes"].cpu(), pred["labels"].cpu(), pred["scores"].cpu()):
    if score < .6: continue
    x1,y1,x2,y2 = box.tolist()
    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,linewidth=2))
    ax.text(x1,y1, f"{categories[int(label)]} {float(score):.2f}", color="white", bbox=dict(facecolor="green",alpha=.7))
ax.axis("off"); plt.show()

## Compare with YOLO
- Faster R-CNN is a **two-stage** detector and is often useful when accuracy is prioritized.
- YOLO is a **one-stage** detector and is often chosen for speed.
- Actual speed/accuracy depends on hardware, model size, dataset, and thresholds.

**Class task:** change confidence threshold from `0.6` to `0.3`.